In [1]:
'''
We have entry coordinates and ventral distance that successfully target a brain region at
a known pitch and zero roll.

Estimate the new roll and rotate to a different pitch.

Given these changes, estimate the new entry coordinates and ventral distance to hit the same target.

TODO update to allow angled probe
'''
import numpy as np

In [ ]:
'''
Basic methodology (todo):
1. known stereotax-vertical vector at a given head angle v_stereo surface to target
2. use AP and DV to calculate the vector from target to lambda v_l_theta (angle, distance)
3. rotate v_l_theta to a new head angle to get v_l_phi
4. calculate rotation needed to rotate v_l_phi to stereotax-vertical v_stereo
5. use this angle to determine AP, DV

notes:
- AP is in stereo-level
- consider DV relative to brain surface (approximated as a plane) vs relative to stereo-level




'''

In [32]:
def estimate_roll(dv_left, dv_right, ml_offset):
    '''
    Given DV offset between 2 points equidistant from the midline
    estimate the roll in radians.
    '''
    ml_diff = np.round(dv_left-dv_right, 2)
    return np.arctan2(ml_diff, 2*ml_offset)

In [33]:
''' Functions to rotate along each axis - todo check signs '''
def rotate_AP(theta):
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)
    rot_mat = np.asarray([
        [1, 0, 0],
        [0, cos_theta, -sin_theta],
        [0, sin_theta, cos_theta]
    ])
    return rot_mat

def rotate_ML(theta):
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)
    rot_mat = np.asarray([
        [cos_theta, 0, sin_theta],
        [0, 1, 0],
        [-sin_theta, 0, cos_theta]
    ])
    return rot_mat

def rotate_DV(theta):
    cos_theta = np.cos(theta)
    sin_theta = np.sin(theta)
    rot_mat = np.asarray([
        [cos_theta, -sin_theta, 0],
        [sin_theta, cos_theta, 0],
        [0, 0, 1]
    ])
    return rot_mat

In [34]:
def convert_head_to_stereo(pitch, roll):
    '''
    Given the pitch and roll of the head, map vectors from
    head reference frame to stereotaxic reference frame.
    
    Roll rotates about the AP axis (right/left side are not level)
    Pitch rotates about the ML axis (front/back are relatively offset)
    
    R is the rotation matric mapping vectors from head-frame to stereo-frame
    '''
    R = rotate_AP(roll) @ rotate_ML(pitch)
    return R

def probe_dir_brain(pitch, roll, v_stereo=np.asarray([0, 0, 1])):
    '''
    Given the pitch and roll of the head, map vectors from
    stereotaxic reference frame to head reference frame.
    
    Default is vertically mounted probe.
    '''
    R = convert_head_to_stereo(pitch, roll)
    
    # convert stereo to head vector
    v_brain = R.T @ v_stereo
    v_brain = v_brain / np.linalg.norm(v_brain)
    
    # make sure DV is positive
    if v_brain[2] < 0:
        v_brain = -v_brain
        
    return v_brain

In [35]:
def get_target_loc(AP_entry, ML_entry, DV_probe, pitch_deg, roll_rad):
    '''
    Calculate target location [AP, ML, DV] in 3D brain coordinates.
    
    AP_entry, ML_entry : float
        in mm, AP and ML coordinates of entry point
    DV_probe : float
        in mm, depth of probe insertion (+ is ventral)
    pitch_deg : int
        beak bar angle
    roll_rad : float
        computed from ML difference using estimate_roll(...)
    '''
    # for clarity, convert beak bar angle to histology level
    hist_deg = pitch_deg - 40
    
    # convert head angle to radians
    hist_rad = np.deg2rad(hist_deg)
    
    # get entry point and vector direction to target
    E = np.asarray([AP_entry, ML_entry, 0.0])
    v_brain = probe_dir_brain(hist_rad, roll_rad)
    
    # get target location in the brain
    T = E + (DV_probe*v_brain)
    
    return T

In [36]:
def get_new_coords(target_loc, pitch_deg, roll_rad):
    '''
    Given a target location in the brain [AP, ML, DV]
    and a measured ML roll and AP pitch, calculate the 
    entry point and depth for (vertical) probe insertion.
    
    TODO update to allow angled probe
    '''
    # convert beak bar angle to histology level
    hist_deg = pitch_deg - 40
    
    # convert head angle to radians
    hist_rad = np.deg2rad(hist_deg)
    
    # ensure target is properly formatted
    T = np.asarray(target_loc, dtype=float)
    
    # new vector direction from target to surface
    v_brain = probe_dir_brain(hist_rad, roll_rad)
    
    # calculate the vertical distance from the target to the surface
    surface_dist = T[2] / v_brain[2]
    
    # get the new entry vector
    E = T - (surface_dist*v_brain)
    
    return {
        "entry_AP": np.round(E[0], 2),
        "entry_ML": np.round(E[1], 2),
        "travel_DV": np.round(surface_dist, 2)
    }

In [37]:
''' Calculate the 3D brain location of LHy (empirically determined) '''
# known insertion params
AP = 1.4
ML = 0.48
DV = 5.8

# tested head angle
pitch = 49 # degrees
roll = 0 

lhy_loc = get_target_loc(AP_entry=AP, 
                         ML_entry=ML, 
                         DV_probe=DV, 
                         pitch_deg=90-pitch, 
                         roll_rad=0)

In [38]:
rough_loc = np.round(lhy_loc, 2)
print(f"empirically estimated LHy location in the brain: AP = {rough_loc[0]}, ML = {rough_loc[1]}, DV = {rough_loc[2]}")

empirically estimated LHy location in the brain: AP = 1.3, ML = 0.48, DV = 5.8


In [39]:
''' Calculate the new probe targeting coordinates '''
# estimate ML roll given DV offset L/R
DV_left = input("DV left = ")
DV_right = input("DV right = ")
ML_offset = input("ML offset = ")
roll_new = estimate_roll(float(DV_left), float(DV_right), ml_offset=float(ML_offset))

# new head angle
pitch = input("beak bar angle = ") # degrees

# get new targeting coords
new_coords = get_new_coords(lhy_loc, 90-int(pitch), roll_new)

print("updated targeting coordinates:")
print(new_coords)

DV left = 0
DV right = 0
ML offset = 2
beak bar angle = 30
updated targeting coordinates:
{'entry_AP': 3.41, 'entry_ML': 0.48, 'travel_DV': 6.17}
